In [0]:
%run ../00-common/01-env-variables

In [0]:
from pyspark.sql import functions as F

In [0]:
results_df = (
    spark.table(f"{catalog_name}.{silver_schema}.results")
            .withColumn('session_type' , F.lit('RACE'))
            .drop('race_name' , 'race_date' , 'ingestion_timestamp' , 'source_file')
)


In [0]:
sprints_df = (
    spark.table(f"{catalog_name}.{silver_schema}.sprints")
            .withColumn('session_type' , F.lit('SPRINT'))
            .drop('race_name' , 'race_date' , 'ingestion_timestamp' , 'source_file')
)

In [0]:
results_sprints_df = results_df.unionByName(sprints_df)


In [0]:
final_results_sprints_df = (
    results_sprints_df
    .withColumn('is_win' , F.col('finish_positiom') == 1)
    .withColumn('is_podium' , F.col('finish_positiom').between(1,3))
    .withColumn('has_points' , F.col("points")> 0)

)

In [0]:
final_results_sprints_df.write.mode('overwrite').saveAsTable(f"{catalog_name}.{gold_schema}.fact_session_results")